# Sensitivity at Fixed Specificity (Sens@XXSpec) — TM Comparison

Compare ensembles on the **TM (mixed) frozen test set** only. TM is the primary holdout (T5 ∪ TB ∪ TC, binary view) and is used for deployment decisions.

**Metrics:** AUROC, Sens@80Spec, Sens@85Spec, Sens@90Spec, Brier, ECE, plus accuracy/sensitivity/specificity at 0.5 threshold.

**Ensembles (edit the list below to add more):**
- Production binary (assets)
- Production 5-class (assets)
- Progressive_K5 binary
- Progressive_K5 5-class

Later: add histopathology-backbone ensembles by appending entries with the same structure (model_dir or onnx_paths, model_type).

## 1. Paths and configuration

In [5]:
import sys
import json
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent.parent.parent  # repo root (notebooks -> training -> ml -> repo)
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"

# Specificity targets for Sens@XXSpec
SPEC_TARGETS = [0.80, 0.85, 0.90]

# Ensemble definitions: name, model_path (dir with ensemble_model_1/2/3.onnx) or onnx_paths (list of 3 for custom names), model_type
# model_type: "binary_onnx_ensemble" (dir with ensemble_model_*.onnx), "5class_onnx_ensemble", "binary_onnx_ensemble_custom" (onnx_paths list)
ENSEMBLES = [
    {
        "name": "Production binary",
        "onnx_paths": [
            REPO_ROOT / "assets" / "binary_ensemble_mode_1.onnx",
            REPO_ROOT / "assets" / "binary_ensemble_mode_2.onnx",
            REPO_ROOT / "assets" / "binary_ensemble_mode_3.onnx",
        ],
        "model_type": "binary_onnx_ensemble_custom",
    },
    {
        "name": "Production 5-class",
        "model_dir": REPO_ROOT / "assets",
        "model_type": "5class_onnx_ensemble",
    },
    {
        "name": "Progressive_K5 binary",
        "model_dir": NOTEBOOK_DIR / "results" / "freeze_comparison" / "freeze_comparison_20260208_121618" / "onnx",
        "model_type": "binary_onnx_ensemble",
    },
    {
        "name": "Progressive_K5 5-class",
        "model_dir": NOTEBOOK_DIR / "results" / "improved_5class" / "improved_5class_20260209_172821" / "onnx",
        "model_type": "5class_onnx_ensemble",
    },
    # Add histology-backbone ensembles later, e.g.:
    # {"name": "Histology progressive_K5 binary", "model_dir": NOTEBOOK_DIR / "results" / "histology_transfer" / "run_xxx" / "onnx", "model_type": "binary_onnx_ensemble"},
]

OUTPUT_JSON = NOTEBOOK_DIR / "results" / "sens_at_spec_tm_comparison.json"
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"FROZEN_DIR: {FROZEN_DIR}")
print(f"SPEC_TARGETS: {SPEC_TARGETS}")
print(f"Ensembles: {[e['name'] for e in ENSEMBLES]}")

REPO_ROOT: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier
FROZEN_DIR: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
SPEC_TARGETS: [0.8, 0.85, 0.9]
Ensembles: ['Production binary', 'Production 5-class', 'Progressive_K5 binary', 'Progressive_K5 5-class']


## 2. Load TM and evaluation helpers

In [6]:
import numpy as np
from sklearn.metrics import roc_auc_score, brier_score_loss

sys.path.insert(0, str(NOTEBOOK_DIR))
from frozen_test_sets import get_tm_paths_labels_and_sources_v2
from evaluate_on_frozen_sets import (
    load_and_preprocess_image_onnx_ensemble,
    sensitivity_at_specificity,
    expected_calibration_error,
    ENSEMBLE_WEIGHTS,
    run_predictions_binary_onnx_ensemble,
    run_predictions_5class_onnx_ensemble,
)

try:
    import onnxruntime as ort
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False

tm_paths, tm_labels, tm_sources = get_tm_paths_labels_and_sources_v2(FROZEN_DIR)
tm_true = np.array(tm_labels)
print(f"TM: n={len(tm_true)}  cancer={tm_true.sum()}  non-cancer={(tm_true==0).sum()}")

TM: n=779  cancer=402  non-cancer=377


In [7]:
def run_binary_onnx_ensemble_from_paths(onnx_paths, paths, labels_binary, batch_size=32, image_size=(224, 224)):
    """Run binary ensemble from explicit list of 3 ONNX paths (e.g. assets binary_ensemble_mode_*.onnx)."""
    if not HAS_ONNX:
        raise RuntimeError("onnxruntime required")
    sessions = []
    input_name = None
    for p in onnx_paths:
        p = Path(p)
        if not p.exists():
            raise FileNotFoundError(f"{p}")
        sess = ort.InferenceSession(str(p))
        sessions.append(sess)
        if input_name is None:
            input_name = sess.get_inputs()[0].name
    preds, probs = [], []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i : i + batch_size]
        batch_arrays = [load_and_preprocess_image_onnx_ensemble(p, image_size) for p in batch_paths]
        batch_input = np.concatenate(batch_arrays, axis=0)
        all_preds = []
        for sess, w in zip(sessions, ENSEMBLE_WEIGHTS):
            out = sess.run(None, {input_name: batch_input})[0]
            all_preds.append(out * w)
        ensemble_logits = np.sum(all_preds, axis=0)
        pred_binary = np.argmax(ensemble_logits, axis=1)
        exp_logits = np.exp(ensemble_logits - np.max(ensemble_logits, axis=1, keepdims=True))
        prob_cancer = (exp_logits[:, 1] / exp_logits.sum(axis=1)).astype(np.float64)
        preds.extend(pred_binary.tolist())
        probs.extend(prob_cancer.tolist())
    return np.array(preds), np.array(probs)


def metrics_tm(true, pred, prob, spec_targets=(0.80, 0.85, 0.90)):
    """Compute metrics including Sens@XXSpec for given specificity targets."""
    acc = float(np.mean(pred == true))
    n_pos = (true == 1).sum()
    n_neg = (true == 0).sum()
    sens = float(np.mean(pred[true == 1] == 1)) if n_pos else 0.0
    spec = float(np.mean(pred[true == 0] == 0)) if n_neg else 0.0
    out = {"n": int(len(true)), "accuracy": acc, "sensitivity": sens, "specificity": spec}
    if prob is not None and len(prob) == len(true) and np.unique(true).size >= 2:
        out["auroc"] = float(roc_auc_score(true, prob))
        out["brier"] = float(brier_score_loss(true, prob))
        out["ece"] = expected_calibration_error(true, prob)
        for t in spec_targets:
            out[f"sens_at_{int(t*100)}spec"] = sensitivity_at_specificity(true, prob, t)
    return out

## 3. Run each ensemble on TM and compute metrics

In [8]:
results = []
image_size = (224, 224)
batch_size = 32

for cfg in ENSEMBLES:
    name = cfg["name"]
    print(f"Running: {name}...")
    try:
        if cfg["model_type"] == "binary_onnx_ensemble_custom":
            pred, prob = run_binary_onnx_ensemble_from_paths(
                cfg["onnx_paths"], tm_paths, tm_labels, batch_size=batch_size, image_size=image_size
            )
        elif cfg["model_type"] == "binary_onnx_ensemble":
            pred, prob = run_predictions_binary_onnx_ensemble(
                cfg["model_dir"], tm_paths, tm_labels, batch_size=batch_size, image_size=image_size
            )
        elif cfg["model_type"] == "5class_onnx_ensemble":
            pred, prob = run_predictions_5class_onnx_ensemble(
                cfg["model_dir"], tm_paths, tm_labels, batch_size=batch_size, image_size=image_size
            )
        else:
            print(f"  Skip unknown model_type: {cfg['model_type']}")
            continue
        m = metrics_tm(tm_true, pred, prob, spec_targets=SPEC_TARGETS)
        m["ensemble_name"] = name
        results.append(m)
        print(f"  AUROC={m.get('auroc', 0):.4f}  S@80={m.get('sens_at_80spec', 0)*100:.1f}%  S@85={m.get('sens_at_85spec', 0)*100:.1f}%  S@90={m.get('sens_at_90spec', 0)*100:.1f}%")
    except FileNotFoundError as e:
        print(f"  FileNotFoundError: {e}")
        results.append({"ensemble_name": name, "error": str(e)})
    except Exception as e:
        print(f"  Error: {e}")
        results.append({"ensemble_name": name, "error": str(e)})

print("Done.")

Running: Production binary...
  AUROC=0.9277  S@80=88.6%  S@85=84.1%  S@90=81.6%
Running: Production 5-class...
  AUROC=0.8731  S@80=78.9%  S@85=76.4%  S@90=65.2%
Running: Progressive_K5 binary...
  AUROC=0.9291  S@80=88.6%  S@85=84.8%  S@90=78.9%
Running: Progressive_K5 5-class...
  AUROC=0.8568  S@80=73.9%  S@85=68.9%  S@90=58.7%
Done.


## 4. Summary table

In [9]:
def row(m):
    if "error" in m:
        return f"{m['ensemble_name']:<30}  ERROR: {m['error']}"
    return (
        f"{m['ensemble_name']:<30}  n={m['n']}  "
        f"AUROC={m.get('auroc', 0):.4f}  "
        f"S@80={m.get('sens_at_80spec', 0)*100:.1f}%  S@85={m.get('sens_at_85spec', 0)*100:.1f}%  S@90={m.get('sens_at_90spec', 0)*100:.1f}%  "
        f"Brier={m.get('brier', 0):.4f}  ECE={m.get('ece', 0):.4f}  "
        f"Acc={m.get('accuracy', 0)*100:.1f}%  Sens={m.get('sensitivity', 0)*100:.1f}%  Spec={m.get('specificity', 0)*100:.1f}%"
    )

print("TM (mixed frozen set) — Sensitivity at fixed specificity")
print("=" * 120)
for m in results:
    print(row(m))
print("=" * 120)

TM (mixed frozen set) — Sensitivity at fixed specificity
Production binary               n=779  AUROC=0.9277  S@80=88.6%  S@85=84.1%  S@90=81.6%  Brier=0.1096  ECE=0.3870  Acc=84.5%  Sens=85.1%  Spec=83.8%
Production 5-class              n=779  AUROC=0.8731  S@80=78.9%  S@85=76.4%  S@90=65.2%  Brier=0.1675  ECE=0.4206  Acc=80.4%  Sens=77.4%  Spec=83.6%
Progressive_K5 binary           n=779  AUROC=0.9291  S@80=88.6%  S@85=84.8%  S@90=78.9%  Brier=0.1099  ECE=0.3775  Acc=84.6%  Sens=85.6%  Spec=83.6%
Progressive_K5 5-class          n=779  AUROC=0.8568  S@80=73.9%  S@85=68.9%  S@90=58.7%  Brier=0.1608  ECE=0.3352  Acc=76.6%  Sens=74.1%  Spec=79.3%


## 5. Save results and optional pandas table

In [10]:
save_data = {
    "spec_targets": SPEC_TARGETS,
    "tm_n": int(len(tm_true)),
    "ensembles": [e["name"] for e in ENSEMBLES],
    "results": results,
}
with open(OUTPUT_JSON, "w") as f:
    json.dump(save_data, f, indent=2)
print(f"Saved: {OUTPUT_JSON}")

try:
    import pandas as pd
    rows = []
    for m in results:
        if "error" in m:
            continue
        rows.append({
            "Ensemble": m["ensemble_name"],
            "n": m["n"],
            "AUROC": m.get("auroc"),
            "S@80Spec": m.get("sens_at_80spec"),
            "S@85Spec": m.get("sens_at_85spec"),
            "S@90Spec": m.get("sens_at_90spec"),
            "Brier": m.get("brier"),
            "ECE": m.get("ece"),
            "Accuracy": m.get("accuracy"),
            "Sensitivity": m.get("sensitivity"),
            "Specificity": m.get("specificity"),
        })
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    print("pandas not installed; skip DataFrame display.")

Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/sens_at_spec_tm_comparison.json


,Ensemble,n,AUROC,S@80Spec,S@85Spec,S@90Spec,Brier,ECE,Accuracy,Sensitivity,Specificity
0,Production binary,779,0.927689,0.885572,0.840796,0.815920,0.109647,0.386979,0.844673,0.850746,0.838196
1,Production 5-class,779,0.873101,0.788557,0.763682,0.651741,0.167538,0.420621,0.803594,0.773632,0.835544
2,Progressive_K5 binary,779,0.929081,0.885572,0.848259,0.788557,0.109850,0.377508,0.845956,0.855721,0.835544
3,Progressive_K5 5-class,779,0.856797,0.738806,0.689055,0.587065,0.160811,0.335168,0.766367,0.741294,0.793103
